In [6]:
import sys
import os

# tambahin root project ke path
sys.path.append(os.path.abspath(".."))

In [7]:
from utils.dataloader import get_dataloader

In [8]:
BASE_DIR = os.path.abspath("..")

train_path = os.path.join(BASE_DIR, "data/WLBisindo/split/train")
val_path   = os.path.join(BASE_DIR, "data/WLBisindo/split/val")
test_path  = os.path.join(BASE_DIR, "data/WLBisindo/split/test")

print(train_path)

D:\Softwares\AnacondaProjects\GestureSequenceCNN\data/WLBisindo/split/train


In [9]:
import os

print("Train exists:", os.path.exists(train_path))
print("Val exists:", os.path.exists(val_path))
print("Test exists:", os.path.exists(test_path))

Train exists: True
Val exists: True
Test exists: True


In [10]:
train_loader = get_dataloader(train_path)
val_loader   = get_dataloader(val_path, shuffle=False)
test_loader  = get_dataloader(test_path, shuffle=False)

In [11]:
for x, y in train_loader:
    print("Input shape :", x.shape)  # (B, 20, 3, 224, 224)
    print("Label shape :", y.shape) # (B,)
    print("Label sample:", y[:5])
    break

Input shape : torch.Size([8, 20, 3, 224, 224])
Label shape : torch.Size([8])
Label sample: tensor([ 2, 25, 25, 14,  7])


In [ ]:
import os
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter

# buat folder run unik
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

BASE_DIR = os.path.abspath("..")
log_dir = os.path.join(BASE_DIR, "outputs/logs/cnn_lstm", f"run_{timestamp}")

os.makedirs(log_dir, exist_ok=True)

writer = SummaryWriter(log_dir)

print("Log dir:", log_dir)

In [ ]:
log_file = os.path.join(log_dir, "log.txt")

def log_message(msg):
    print(msg)
    with open(log_file, "a") as f:
        f.write(msg + "\n")

In [2]:
# Tensor Board
# type in terminal = tensorboard --logdir=outputs/logs
# http://localhost:6006

In [ ]:
# ambil jumlah kelas dari dataset
NUM_CLASSES = len(train_loader.dataset.label_map)

print("Num classes:", NUM_CLASSES)

In [ ]:
from models.cnn_lstm import CNN_LSTM

model = CNN_LSTM(num_classes=NUM_CLASSES)

x, y = next(iter(train_loader))
out = model(x)

print(out.shape)

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = CNN_LSTM(num_classes=NUM_CLASSES).to(device)

for param in model.encoder.cnn.parameters():
    param.requires_grad = False

criterion = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=2,
    factor=0.5
)

In [ ]:
def compute_accuracy(preds, labels):
    preds = torch.argmax(preds, dim=1)
    correct = (preds == labels).sum().item()
    return correct / len(labels)

In [ ]:
EPOCHS = 10
patience = 3
counter = 0
best_val_loss = float("inf")

for epoch in range(EPOCHS):

    # ===== TRAIN =====
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0

    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        train_loss += loss.item()

        preds = torch.argmax(outputs, dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_loss /= len(train_loader)
    train_acc = train_correct / train_total


    # ===== VALIDATION =====
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            outputs = model(x)
            loss = criterion(outputs, y)

            val_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

    val_loss /= len(val_loader)
    val_acc = val_correct / val_total


    # ===== LOGGING =====
    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Loss/val", val_loss, epoch)

    writer.add_scalar("Accuracy/train", train_acc, epoch)
    writer.add_scalar("Accuracy/val", val_acc, epoch)

    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar("LR", current_lr, epoch)

    log_message(f"Epoch {epoch+1}/{EPOCHS}")
    log_message(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    log_message(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")
    log_message(f"LR: {current_lr}")
    log_message("-" * 40)


    # ===== SCHEDULER =====
    scheduler.step(val_loss)


    # ===== EARLY STOPPING =====
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0

        torch.save(model.state_dict(), os.path.join(log_dir, "best_model.pth"))
        log_message("✅ Best model saved!")

    else:
        counter += 1
        log_message(f"EarlyStopping counter: {counter}/{patience}")

        if counter >= patience:
            log_message("🛑 Early stopping triggered!")
            break


# SAVE LAST MODEL
torch.save(model.state_dict(), os.path.join(log_dir, "last_model.pth"))

In [ ]:
model.eval()

test_correct = 0
test_total = 0

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        outputs = model(x)
        preds = torch.argmax(outputs, dim=1)

        test_correct += (preds == y).sum().item()
        test_total += y.size(0)

test_acc = test_correct / test_total

log_message(f"Test Accuracy: {test_acc:.4f}")
print("Test Accuracy:", test_acc)